In [1]:
import sys
from dataclasses import dataclass

import numpy as np
import polars as pl
from scipy.integrate import solve_ivp
from tqdm.auto import trange

sys.path.append("/app")
from src.polars_utils import random_split

### Preparation

In [2]:
@dataclass
class PendulumParameters:
    """Parameter class for Pendulum simulation."""

    st: float = 0  # Start time (s)
    et: float = 10  # Mean end time (s)
    ts: float = 0.1  # Time step (s)
    g: float = 9.81  # Acceleration due to gravity (m/s^2)
    m: float = 1  # Mass of bob (kg)
    l_min: float = 2
    l_max: float = 4
    b_min: float = 0.2
    b_max: float = 0.5
    n: int = 32  # length of sequence
    alpha: float = 0.5

In [3]:
# data generation from https://arxiv.org/abs/2401.15935


def simulate_poisson(len: int, rate: float):
    """Simulate the Poisson process."""
    inter_arrival = np.random.exponential(1 / rate, size=len)
    return np.cumsum(inter_arrival)


def create_sample(params: PendulumParameters, regular: bool):
    """Generate a Pendulum dataset sample.

    inspired by https://skill-lync.com/student-projects/Simulation-of-a-Simple-Pendulum-on-Python-95518.
    """
    # main

    length = np.random.uniform(params.l_min, params.l_max)
    theta1_ini = np.random.uniform(0, 2 * np.pi)  # Initial angular displacement (rad)
    theta2_ini = np.random.uniform(-np.pi, np.pi)  # Initial angular velocity (rad/s)
    theta_ini = [theta1_ini, theta2_ini]

    b = np.random.uniform(params.b_min, params.b_max)
    if regular:
        t_ir = np.linspace(params.st, params.et, params.n)
    else:
        t_ir = simulate_poisson(params.n, params.n / params.et)

    def sim_pen_eq(_, theta):
        dtheta2_dt = (-b / params.m) * theta[1] + (-params.g / length) * np.sin(
            theta[0]
        )
        dtheta1_dt = theta[1]
        return [dtheta1_dt, dtheta2_dt]

    theta12 = solve_ivp(sim_pen_eq, (t_ir[0], t_ir[-1]), theta_ini, t_eval=t_ir)
    theta1 = theta12.y[0, :]
    return theta1, t_ir, b


def create_dataset(
    train: int, val: int, test: int, params: PendulumParameters, regular: bool
):
    """Create a Pendulum dataset of the specified size."""
    data = []
    for _ in trange(train + val + test):
        angles, time, target = create_sample(params, regular=regular)
        data.append({"target": target, "time": time.tolist(), "theta": angles.tolist()})

    df = pl.DataFrame(data).with_columns(
        pl.col("time").list.eval(
            pl.duration(nanoseconds=pl.first() * 10**9) / pl.duration(seconds=4)
        )
    )

    df = random_split(df, train=train, val=val, test=test)
    return df

### Generation

The angles pendulum dataset:

In [5]:
df = create_dataset(1024, 256, 256, PendulumParameters(), regular=False)
df.write_parquet("/mnt/data/preprocessed/pendulum_angles.parquet")

  0%|          | 0/1536 [00:00<?, ?it/s]

In [6]:
df = create_dataset(1024, 256, 256, PendulumParameters(), regular=True)
df.write_parquet("/mnt/data/preprocessed/pendulum_angles_regular.parquet")

  0%|          | 0/1536 [00:00<?, ?it/s]